# 第 3 章 消息：对话里的不同角色

## 一句话不够用

上一章的 `call` 只能传入一句话。可小明对学习助手有更多的要求：

- 它应该知道自己是谁：是这本书的学习助手，面对的是编程新手。
- 它应该有固定的风格：比如用中文、讲得简单一点。
- 它最好还能看到之前聊过的内容。

这些都不是“一句话”能装下的。真实的对话，由很多条**消息**组成，每条消息都有一个**角色**，表示这句话是谁说的。这一章，我们就来认识对话里的三种角色，并用 Python 类把消息包装起来。

## 对话里的三种角色

| 角色 | 是谁说的 | 例子 |
|---|---|---|
| system（系统） | 开发者，也就是我们 | “你是一个学习助手，回答要简短，适合新手。” |
| user（用户） | 使用者，比如小明 | “什么是变量？” |
| assistant（助手） | 大模型 | “变量就像一个贴了名字的盒子……” |

一段对话发给大模型时，大致是这个样子：

```
system:    你是一个学习助手，回答要简短，适合新手。
user:      什么是变量？
assistant: 变量就像一个贴了名字的盒子，可以用来存放数据。
user:      那常量呢？
```

user 和 assistant 很好理解，就是一问一答。最特别的是 **system**。

## 系统消息（system message）

系统消息是开发者写给大模型的“幕后说明”，用来设定它**是谁、做什么、怎么做**。使用者一般看不到它，但它对整段对话都有效。

系统消息里通常可以写这些内容：

- **身份**：你是谁。比如“你是《从大模型到智能体》这本书的学习助手”。
- **任务**：你要做什么。比如“帮助初学者学习如何构建 AI 智能体”。
- **风格**：怎么回答。比如“用中文回答，尽量简短，多举例子”。
- **规则**：不能做什么。比如“不要直接给出练习题的答案，要引导学生自己思考”。

写系统消息有几个小技巧：

- **具体**。“回答简短”不如“每次回答不超过三句话”。
- **少而精**。把最重要的要求写清楚就行，写得太长太杂，模型反而抓不住重点。
- **一次只改一处**。效果不理想时，改一个地方，看看效果，再改下一个地方。

同样的模型，换一条系统消息，就能变成一个风格完全不同的助手。我们的学习助手，就从一条系统消息开始。

## 为什么要把消息包装成类

我们完全可以直接用字典来表示消息，比如 `{"role": "user", "content": "什么是变量？"}`。那为什么还要专门写一组类呢？

**第一，不同厂商的格式不一样。** 拿系统消息来说：

- OpenAI 把系统消息当成普通的一条消息，放在 `messages` 列表的最前面，`role` 是 `"system"`。
- Claude 的 `messages` 里没有 system 角色，系统消息要通过单独的 `system` 参数传进去。

如果程序里直接拼字典，就得处处判断“现在用的是哪家”。把消息包装成我们自己的类，程序的其他部分只和这些类打交道；怎么转换成各家的格式，交给上一章的 `LLMProvider` 去处理。

**第二，方便以后扩展。** 现在的消息只有一段文字。往后，消息里还会出现工具调用（第 9 章）、模型的思考过程（第 18 章），还要能保存到文件里（第 7 章）。有了自己的类，这些功能都可以一点点加进去，而不用改动程序的其他部分。

接下来看代码。先导入需要的模块：

In [ ]:
import os
from abc import ABC, abstractmethod
from enum import Enum
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

## 消息类

首先用一个**枚举**（`Enum`）列出三种角色。枚举就是一组固定的选项，比直接写字符串更不容易写错：

In [ ]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

然后是所有消息的“父类” `Message`。它也是一个抽象类，规定了每条消息都要有两样东西：

- `msg_type`：这条消息的角色。
- `summary()`：返回这条消息的文字内容。不管是哪种消息，都可以用 `summary()` 拿到它的文字。

`__repr__` 决定了用 `print` 打印一条消息时显示成什么样子，方便我们调试。

In [ ]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

三种角色各有一个子类。它们的结构都一样：在 `__init__` 里设置自己的角色，再保存消息的文字。为了让代码读起来更清楚，文字字段的名字各不相同：系统消息叫 `description`（描述），用户消息叫 `request`（请求），助手消息叫 `content`（内容）。

In [ ]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [ ]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [ ]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

现在可以写出学习助手的系统消息了。它会伴随我们整本书。这里用英文写，大模型对中英文都理解得很好，你也可以换成中文。

In [ ]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")
user_message = UserMessage("你好，我叫小明，正在自学编程。请用两三句话介绍一下你自己。")

## 让 Provider 认识消息类

`LLMProvider` 的 `call` 也要跟着升级：不再是传入一句话、返回一段文字，而是：

- 传入一条系统消息 `system` 和一个消息列表 `messages`；
- 返回一条 `AssistantMessage`。

这样，进出 `call` 的都是我们自己的消息类，程序的其他部分完全不需要知道各家 API 的格式。

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

`OpenAIProvider` 多了两个“翻译”方法：

- `_message_schema`：把我们的消息列表，翻译成 OpenAI 要求的格式。用 `isinstance` 判断每条消息是哪个类，再转换成对应的 `role`。
- `_response_schema`：把 OpenAI 返回的结果，翻译回我们的 `AssistantMessage`。

注意 `call` 的第一行：如果有系统消息，就把它放到消息列表的最前面。这正是 OpenAI 处理系统消息的方式。`max_tokens` 对应 OpenAI 的参数 `max_completion_tokens`，表示最多生成多少 token。

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        return AssistantMessage(message.content or "")

In [ ]:
provider = OpenAIProvider()
message = provider.call(system_message, [user_message])
print(message.summary())

`ClaudeProvider` 也有同样的两个翻译方法，但处理系统消息的方式不一样：系统消息不放进 `messages`，而是放到单独的 `system` 参数里。所以它的 `_message_schema` 只需要处理用户消息和助手消息。

另外，Claude 返回的 `response.content` 是一个列表，里面可能有好几段内容。我们把其中所有文字（`type == "text"`）拼起来，作为回答。

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        return AssistantMessage(text)

In [ ]:
provider = ClaudeProvider()
message = provider.call(system_message, [user_message])
print(message.summary())

同样一句话，两家的处理方式不同，但对我们来说，调用的代码一模一样。

## 试一试系统消息的作用

最后做个小实验：同一个问题，配上两条不同的系统消息，看看回答有什么不同。

In [ ]:
question = UserMessage("什么是变量？")
for style in ["用一句话回答，适合完全没学过编程的人。",
              "像一位耐心的老师那样回答，并举一个 Python 的小例子。"]:
    reply = provider.call(SystemMessage(style), [question])
    print(f"--- system: {style}\n{reply.summary()}\n")

模型、问题都没变，只换了系统消息，回答的长短和风格就完全不一样了。

## 动手试试

- 修改学习助手的系统消息，比如加上“每次回答结尾，提一个问题帮学生检查自己是否理解”，看看效果。
- 消息列表里也可以放之前的对话。试试传入 `[UserMessage("我叫小明"), AssistantMessage("你好，小明！"), UserMessage("我叫什么名字？")]`，模型能答出来吗？想一想为什么。

## 小结

- 对话由多条**消息**组成，每条消息有一个**角色**：system（开发者的设定）、user（使用者）、assistant（大模型）。
- **系统消息**设定助手的身份、任务、风格和规则，对整段对话都有效。写得具体、简洁，效果最好。
- 我们把消息包装成 `SystemMessage`、`UserMessage`、`AssistantMessage` 三个类：程序只和这些类打交道，各家的格式由 Provider 负责翻译，以后加新功能也更方便。
- OpenAI 把系统消息放在 `messages` 里，Claude 用单独的 `system` 参数，这些差别都藏在了 Provider 里面。

到这里，学习助手已经有了身份和风格，可每次还是只能回答一个问题。下一章，我们写出第一个**智能体循环**，让小明可以和它一直聊下去。